# Parte 2: SQL para Exploración de Datos

En este notebook cargamos los datos en SQLite, creamos la tabla `vendedores` y ejecutamos las consultas del archivo `02_sql_discovery.sql`.

In [4]:
import pandas as pd
import sqlite3
from pathlib import Path
from io import StringIO

csv_path = Path('ventas_techventas.csv')
fixed_lines = []

with open(csv_path, 'r', encoding='latin-1') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Corrección real de las comillas que envuelven registros completos
        if line.startswith('"') and line.endswith('"'):
            line = line[1:-1]
        fixed_lines.append(line)

clean_csv = '\n'.join(fixed_lines)
df = pd.read_csv(StringIO(clean_csv), parse_dates=['fecha'])

df['cantidad'] = df['cantidad'].astype(int)
df['precio_unitario'] = df['precio_unitario'].astype(float)

conn = sqlite3.connect('techventas.db')
df.to_sql('ventas', conn, if_exists='replace', index=False)

vendedores = pd.DataFrame([
    ['Ana López', 'Norte', 8000],
    ['Carlos Ruiz', 'Sur', 7500],
    ['Luis Mora', 'Norte', 8000],
    ['Maria Torres', 'Centro', 7000],
], columns=['vendedor','zona','meta_mensual'])
vendedores.to_sql('vendedores', conn, if_exists='replace', index=False)

conn.close()
df.head(5)

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2,1200.0,0.10,Ana Lï¿½ï¿
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15,25.0,0.00,Carlos Ruiz
2,1003,2024-01-10,C003,DataSolutions,Centro,"Monitor 27""""",Electronica,5,350.0,0.05,Ana Lï¿½ï¿
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3,1200.0,0.15,Carlos Ruiz


## Ejecutar consultas SQL

A continuación se ejecutan las consultas guardadas en `02_sql_discovery.sql` y se muestran los resultados.

In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('techventas.db')

# Se agregaron comillas en las fechas de Q2 y se removió la columna inexistente cliente_nombre en Q6
queries = {
    'Q1': "SELECT DISTINCT producto AS producto_unico FROM ventas ORDER BY producto_unico;",
    'Q2': "SELECT * FROM ventas WHERE fecha BETWEEN '2024-01-01' AND '2024-03-31' AND cantidad > 5 ORDER BY fecha;",
    'Q3': "SELECT vendedor, SUM(cantidad * precio_unitario) AS ingreso_bruto_total FROM ventas GROUP BY vendedor HAVING ingreso_bruto_total > 10000 ORDER BY ingreso_bruto_total DESC;",
    'Q4': "SELECT categoria, COUNT(order_id) AS total_pedidos, SUM(cantidad) AS suma_unidades_vendidas, AVG(precio_unitario) AS precio_unitario_promedio FROM ventas GROUP BY categoria ORDER BY categoria;",
    'Q5': "SELECT v.vendedor, v.zona, SUM(t.cantidad * t.precio_unitario) AS ingreso_total_semestre, v.meta_mensual, ROUND(SUM(t.cantidad * t.precio_unitario) / (v.meta_mensual * 6) * 100, 2) AS porcentaje_cumplimiento FROM ventas AS t JOIN vendedores AS v ON t.vendedor = v.vendedor GROUP BY v.vendedor, v.zona, v.meta_mensual ORDER BY porcentaje_cumplimiento DESC;",
    'Q6': "SELECT cliente_id, ingreso_total FROM (SELECT cliente_id, SUM(cantidad * precio_unitario) AS ingreso_total FROM ventas GROUP BY cliente_id) AS totales WHERE ingreso_total = (SELECT MAX(ingreso_total) FROM (SELECT cliente_id, SUM(cantidad * precio_unitario) AS ingreso_total FROM ventas GROUP BY cliente_id));"
}

for name, sql in queries.items():
    print('---', name, '---')
    print(pd.read_sql(sql, conn).to_string(index=False))
    print()

conn.close()

--- Q1 ---
   producto_unico
              NaN
    Laptop Pro 15
     Monitor 27""
Mouse Inalambrico
    Router WiFi 6
          SSD 1TB
 Teclado Mecanico

--- Q2 ---
 order_id               fecha cliente_id    cliente_nombre region          producto      categoria  cantidad  precio_unitario  descuento     vendedor
     1002 2024-01-07 00:00:00       C002         Innovatek    Sur Mouse Inalambrico    Perifericos        15             25.0       0.00  Carlos Ruiz
     1004 2024-01-12 00:00:00       C001       TechCorp SA  Norte  Teclado Mecanico    Perifericos        10             85.0       0.10    Luis Mora
     1006 2024-01-18 00:00:00       C005 Sistemas Globales  Oeste           SSD 1TB Almacenamiento        20             95.0       0.00 Maria Torres
     1008 2024-01-22 00:00:00       C006          NetPrime  Norte     Router WiFi 6          Redes         8            120.0       0.00    Luis Mora
     1009 2024-01-25 00:00:00       C003     DataSolutions Centro           SSD 1TB